# TheLook E-Commerce — Data Analysis

**Source**: `ntupace.thelook_raw` (BigQuery — available now)

Analyses:
1. Monthly Sales Trends
2. Top-Selling Products
3. Revenue by Category & Price Tier
4. Customer Segmentation (age, channel, gender, country)
5. Customer Lifetime Value (CLV)
6. Key Business KPI Dashboard

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

PROJECT = 'ntupace'
engine  = create_engine(f'bigquery://{PROJECT}/thelook_raw')

def q(sql):
    return pd.read_sql(sql, engine)

fmt_usd = mticker.FuncFormatter(lambda x, _: f'${x:,.0f}')

print('Connected to BigQuery:', PROJECT, '/ thelook_raw')

---
## 1. Monthly Sales Trends
Revenue, gross profit, and order volume by month.

In [ ]:
monthly = q("""
SELECT
    FORMAT_DATE('%Y-%m', o.created_at)     AS sale_month,
    COUNT(oi.id)                           AS total_items,
    COUNT(DISTINCT oi.order_id)            AS total_orders,
    ROUND(SUM(oi.sale_price), 2)           AS revenue,
    ROUND(SUM(oi.sale_price - p.cost), 2)  AS gross_profit,
    ROUND(AVG(oi.sale_price), 2)           AS avg_price
FROM `ntupace.thelook_raw.order_items`  oi
JOIN `ntupace.thelook_raw.orders`       o  ON oi.order_id   = o.order_id
JOIN `ntupace.thelook_raw.products`     p  ON oi.product_id = p.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
  AND o.created_at IS NOT NULL
GROUP BY sale_month
ORDER BY sale_month
""")

x         = range(len(monthly))
step      = max(1, len(monthly) // 14)
tick_idx  = list(range(0, len(monthly), step))
tick_lbl  = monthly['sale_month'].iloc[tick_idx].tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].fill_between(x, monthly['revenue'],      alpha=0.12, color='steelblue')
axes[0].fill_between(x, monthly['gross_profit'], alpha=0.15, color='green')
axes[0].plot(x, monthly['revenue'],      marker='o', color='steelblue', lw=2, ms=4, label='Revenue')
axes[0].plot(x, monthly['gross_profit'], marker='s', color='green',     lw=2, ms=4, label='Gross Profit')
axes[0].set_title('Monthly Revenue vs Gross Profit', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Amount ($)')
axes[0].set_xticks(tick_idx)
axes[0].set_xticklabels(tick_lbl, rotation=45, ha='right')
axes[0].yaxis.set_major_formatter(fmt_usd)
axes[0].legend()

axes[1].bar(x, monthly['total_orders'], color='coral', alpha=0.85)
axes[1].set_title('Monthly Order Volume', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Orders')
axes[1].set_xticks(tick_idx)
axes[1].set_xticklabels(tick_lbl, rotation=45, ha='right')

plt.tight_layout()
plt.savefig('../reports/monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()

peak = monthly.loc[monthly['revenue'].idxmax()]
print(f"Peak revenue month: {peak['sale_month']}  —  ${peak['revenue']:,.0f}")
monthly.tail(6)

---
## 2. Top-Selling Products

In [ ]:
top_products = q("""
SELECT
    p.name    AS product_name,
    p.category,
    p.brand,
    CASE
        WHEN p.retail_price < 20   THEN 'Budget'
        WHEN p.retail_price < 100  THEN 'Mid-range'
        WHEN p.retail_price < 300  THEN 'Premium'
        ELSE 'Luxury'
    END                                        AS price_tier,
    COUNT(oi.id)                               AS units_sold,
    ROUND(SUM(oi.sale_price), 2)               AS total_revenue,
    ROUND(SUM(oi.sale_price - p.cost), 2)      AS total_profit
FROM `ntupace.thelook_raw.order_items`  oi
JOIN `ntupace.thelook_raw.products`     p  ON oi.product_id = p.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
GROUP BY p.name, p.category, p.brand, price_tier
ORDER BY total_revenue DESC
LIMIT 15
""")

tier_colors = {'Budget':'#4CAF50','Mid-range':'#2196F3','Premium':'#FF9800','Luxury':'#9C27B0'}
bar_colors  = [tier_colors.get(t, '#607D8B') for t in top_products['price_tier']]

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(top_products['product_name'], top_products['total_revenue'], color=bar_colors)
ax.set_title('Top 15 Products by Revenue', fontsize=13, fontweight='bold')
ax.set_xlabel('Revenue ($)')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(fmt_usd)
legend_handles = [mpatches.Patch(color=v, label=k) for k, v in tier_colors.items()]
ax.legend(handles=legend_handles, title='Price Tier', loc='lower right')
plt.tight_layout()
plt.savefig('../reports/top_products.png', dpi=150, bbox_inches='tight')
plt.show()
top_products

---
## 3. Revenue by Category & Price Tier

In [ ]:
by_category = q("""
SELECT
    p.category,
    p.department,
    CASE
        WHEN p.retail_price < 20   THEN 'Budget'
        WHEN p.retail_price < 100  THEN 'Mid-range'
        WHEN p.retail_price < 300  THEN 'Premium'
        ELSE 'Luxury'
    END                                                  AS price_tier,
    COUNT(oi.id)                                         AS units_sold,
    ROUND(SUM(oi.sale_price), 2)                         AS revenue,
    ROUND(SUM(oi.sale_price - p.cost), 2)                AS profit,
    ROUND(AVG(SAFE_DIVIDE(oi.sale_price - p.cost,
                          oi.sale_price)) * 100, 1)      AS avg_margin_pct
FROM `ntupace.thelook_raw.order_items`  oi
JOIN `ntupace.thelook_raw.products`     p  ON oi.product_id = p.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
GROUP BY p.category, p.department, price_tier
ORDER BY revenue DESC
""")

cat_agg  = by_category.groupby('category')[['revenue','profit']].sum().reset_index()\
             .sort_values('revenue', ascending=False).head(12)
tier_agg = by_category.groupby('price_tier')['revenue'].sum().reset_index()
tier_order = ['Budget','Mid-range','Premium','Luxury']
tier_agg   = tier_agg.set_index('price_tier').reindex(tier_order).dropna().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=cat_agg, x='revenue', y='category', palette='Blues_r', ax=axes[0])
axes[0].set_title('Top 12 Categories by Revenue', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Revenue ($)')
axes[0].set_ylabel('')
axes[0].xaxis.set_major_formatter(fmt_usd)

pie_colors = [tier_colors[t] for t in tier_agg['price_tier']]
axes[1].pie(tier_agg['revenue'], labels=tier_agg['price_tier'],
            autopct='%1.1f%%', startangle=140, colors=pie_colors)
axes[1].set_title('Revenue Share by Price Tier', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
by_category.head(10)

---
## 4. Customer Segmentation
Revenue by age group, acquisition channel, gender, and top countries.

In [ ]:
seg = q("""
SELECT
    CASE
        WHEN u.age BETWEEN 18 AND 24 THEN '18-24'
        WHEN u.age BETWEEN 25 AND 34 THEN '25-34'
        WHEN u.age BETWEEN 35 AND 44 THEN '35-44'
        WHEN u.age BETWEEN 45 AND 54 THEN '45-54'
        ELSE '55+'
    END                               AS age_bucket,
    u.gender,
    u.traffic_source,
    u.country,
    COUNT(DISTINCT oi.user_id)        AS customers,
    ROUND(SUM(oi.sale_price), 2)      AS revenue,
    ROUND(AVG(oi.sale_price), 2)      AS avg_order_value,
    COUNT(oi.id)                      AS total_items
FROM `ntupace.thelook_raw.order_items`  oi
JOIN `ntupace.thelook_raw.users`        u  ON oi.user_id = u.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
GROUP BY age_bucket, u.gender, u.traffic_source, u.country
ORDER BY revenue DESC
""")

age_rev      = seg.groupby('age_bucket')['revenue'].sum().reset_index().sort_values('age_bucket')
src_rev      = seg.groupby('traffic_source')['revenue'].sum().reset_index().sort_values('revenue', ascending=False)
gender_rev   = seg.groupby('gender')['revenue'].sum().reset_index()
country_rev  = seg.groupby('country')['revenue'].sum().reset_index().sort_values('revenue', ascending=False).head(10)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

axes[0,0].pie(age_rev['revenue'], labels=age_rev['age_bucket'],
              autopct='%1.1f%%', startangle=140,
              colors=sns.color_palette('Set2', len(age_rev)))
axes[0,0].set_title('Revenue by Age Group', fontsize=12, fontweight='bold')

sns.barplot(data=src_rev, x='revenue', y='traffic_source', palette='Purples_r', ax=axes[0,1])
axes[0,1].set_title('Revenue by Acquisition Channel', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Revenue ($)')
axes[0,1].set_ylabel('')
axes[0,1].xaxis.set_major_formatter(fmt_usd)

sns.barplot(data=gender_rev, x='gender', y='revenue',
            palette=['#E91E63','#2196F3'], ax=axes[1,0])
axes[1,0].set_title('Revenue by Gender', fontsize=12, fontweight='bold')
axes[1,0].set_xlabel('')
axes[1,0].set_ylabel('Revenue ($)')
axes[1,0].yaxis.set_major_formatter(fmt_usd)

sns.barplot(data=country_rev, x='revenue', y='country', palette='OrRd_r', ax=axes[1,1])
axes[1,1].set_title('Top 10 Countries by Revenue', fontsize=12, fontweight='bold')
axes[1,1].set_xlabel('Revenue ($)')
axes[1,1].set_ylabel('')
axes[1,1].xaxis.set_major_formatter(fmt_usd)

plt.suptitle('Customer Segmentation Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/customer_segments.png', dpi=150, bbox_inches='tight')
plt.show()
seg.head(8)

---
## 5. Customer Lifetime Value (CLV)
Distribution and median CLV by age group and acquisition channel.

In [ ]:
clv = q("""
SELECT
    u.id                                    AS user_id,
    CONCAT(u.first_name, ' ', u.last_name)  AS full_name,
    CASE
        WHEN u.age BETWEEN 18 AND 24 THEN '18-24'
        WHEN u.age BETWEEN 25 AND 34 THEN '25-34'
        WHEN u.age BETWEEN 35 AND 44 THEN '35-44'
        WHEN u.age BETWEEN 45 AND 54 THEN '45-54'
        ELSE '55+'
    END                                     AS age_bucket,
    u.gender,
    u.country,
    u.traffic_source,
    COUNT(DISTINCT oi.order_id)             AS total_orders,
    ROUND(SUM(oi.sale_price), 2)            AS lifetime_value,
    ROUND(AVG(oi.sale_price), 2)            AS avg_order_value,
    MIN(DATE(oi.created_at))                AS first_purchase,
    MAX(DATE(oi.created_at))                AS last_purchase
FROM `ntupace.thelook_raw.order_items`  oi
JOIN `ntupace.thelook_raw.users`        u  ON oi.user_id = u.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
GROUP BY u.id, u.first_name, u.last_name, age_bucket,
         u.gender, u.country, u.traffic_source
ORDER BY lifetime_value DESC
""")

print('CLV Summary Statistics')
print(clv[['total_orders','lifetime_value','avg_order_value']].describe().round(2))

clv_by_age = clv.groupby('age_bucket')['lifetime_value'].median().reset_index().sort_values('age_bucket')
clv_by_src = clv.groupby('traffic_source')['lifetime_value'].median().reset_index().sort_values('lifetime_value', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(clv['lifetime_value'], bins=50, color='teal', edgecolor='white', alpha=0.85)
median_val = clv['lifetime_value'].median()
axes[0].axvline(median_val, color='red', linestyle='--', lw=2, label=f'Median ${median_val:,.0f}')
axes[0].set_title('CLV Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Lifetime Value ($)')
axes[0].set_ylabel('Customers')
axes[0].xaxis.set_major_formatter(fmt_usd)
axes[0].legend()

sns.barplot(data=clv_by_age, x='age_bucket', y='lifetime_value',
            palette='OrRd', ax=axes[1])
axes[1].set_title('Median CLV by Age Group', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Median CLV ($)')
axes[1].yaxis.set_major_formatter(fmt_usd)

sns.barplot(data=clv_by_src, x='lifetime_value', y='traffic_source',
            palette='Blues_r', ax=axes[2])
axes[2].set_title('Median CLV by Channel', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Median CLV ($)')
axes[2].set_ylabel('')
axes[2].xaxis.set_major_formatter(fmt_usd)

plt.tight_layout()
plt.savefig('../reports/clv_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Customers by Lifetime Value:')
clv[['full_name','country','age_bucket','total_orders','lifetime_value']].head(10)

---
## 6. Key Business KPI Dashboard

In [ ]:
kpis = q("""
SELECT
    COUNT(DISTINCT oi.order_id)                                  AS total_orders,
    COUNT(DISTINCT oi.user_id)                                   AS unique_customers,
    ROUND(SUM(oi.sale_price), 2)                                 AS total_gmv,
    ROUND(SUM(oi.sale_price - p.cost), 2)                        AS total_gross_profit,
    ROUND(AVG(SAFE_DIVIDE(oi.sale_price - p.cost,
                          oi.sale_price)) * 100, 1)              AS avg_gross_margin_pct,
    ROUND(AVG(oi.sale_price), 2)                                 AS avg_item_price,
    ROUND(COUNTIF(oi.status = 'Returned') /
          COUNT(*) * 100, 2)                                     AS return_rate_pct
FROM `ntupace.thelook_raw.order_items`  oi
JOIN `ntupace.thelook_raw.products`     p  ON oi.product_id = p.id
WHERE oi.status IN ('Complete', 'Returned', 'Shipped')
""")

kpi_fmt = {
    'total_orders':         ('Total Orders',      '{:,.0f}',  '#2196F3'),
    'unique_customers':     ('Unique Customers',  '{:,.0f}',  '#4CAF50'),
    'total_gmv':            ('Total GMV',         '${:,.0f}', '#FF9800'),
    'total_gross_profit':   ('Gross Profit',      '${:,.0f}', '#9C27B0'),
    'avg_gross_margin_pct': ('Avg Margin',        '{:.1f}%',  '#00BCD4'),
    'avg_item_price':       ('Avg Item Price',    '${:.2f}',  '#E91E63'),
    'return_rate_pct':      ('Return Rate',       '{:.2f}%',  '#607D8B'),
}

print('=' * 55)
print('  THELOOK E-COMMERCE — KEY BUSINESS METRICS')
print('=' * 55)
for col, (label, fmt, _) in kpi_fmt.items():
    val = kpis[col].iloc[0]
    print(f'  {label:<25}  {fmt.format(val):>15}')
print('=' * 55)

# Visual KPI cards
fig, axes = plt.subplots(1, len(kpi_fmt), figsize=(20, 3))
for ax, (col, (label, fmt, color)) in zip(axes, kpi_fmt.items()):
    val  = kpis[col].iloc[0]
    disp = f'${val/1e6:.1f}M' if (('gmv' in col or 'profit' in col) and val >= 1e6) else fmt.format(val)
    ax.set_facecolor(color)
    ax.text(0.5, 0.62, disp,  ha='center', va='center', fontsize=18,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.28, label, ha='center', va='center', fontsize=9,
            color='white', transform=ax.transAxes)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('TheLook E-Commerce — KPI Dashboard', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('../reports/kpi_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()